# Bulk answer generation

Generate answers for a configurable number of rows from a question JSON file. The default is `Q_S1.json`; change `QUESTION_PATH` for another compatible file.

Reference answers and source-of-truth fields are saved for later evaluation, but only the question text enters retrieval and generation.

## Safety, parallelism, and resume

- `LIVE = False` creates complete dry-run records without OpenRouter calls.
- `LIVE = True` sends paid requests and requires `OPENROUTER_API_KEY`.
- `MAX_WORKERS` controls bounded parallelism. Begin conservatively because rate limits and spend apply.
- There are no automatic API retries.
- The main notebook thread checkpoints each completed worker result.
- To resume, set `RESUME_RUN_DIR` to the exact existing run directory.

In [1]:
import csv
import json
import os
import statistics
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from mobile_rag.answer_generation import GenerationConfig, PROMPT_PATH, PROMPT_VERSION
from mobile_rag.bulk_answer_generation import benchmark_identity, process_question, select_questions
from mobile_rag.context_preparation import ContextBudget
from mobile_rag.environment import openrouter_api_key
from mobile_rag.retrieval import digest, write_json

## Configuration and saved files

Set `NUMBER_OF_QUESTIONS` to a positive integer, such as `5`, to run the first five questions. Set it to `None` to run every question in the selected file. If the number exceeds the available rows, all rows run.

This notebook creates one timestamped directory under `OUTPUT_ROOT` and saves:

- `run_manifest.json`: benchmark, index, prompt, model, context, concurrency, and question-selection configuration.
- `records.jsonl`: one complete record per question, containing the original question row, retrieval hits, neighbor expansion, prepared context, citations, generated result, usage, latency, and statuses.
- `results.csv`: compact review table with reference and generated answers.
- `summary.json`: completion counts, status counts, timing, and missing keys.

Saving is controlled here in the notebook. Worker functions return data and do not save files.

In [2]:
QUESTION_PATH = ROOT / "data/questions/Q_S1.json"  # Change this path when needed.
INDEX_DIR = max((ROOT / "artifacts/03_retrieval_enhanced").glob("*/enhancement_manifest.json")).parent
OUTPUT_ROOT = ROOT / "artifacts/05_2_bulk_answer_generation"
RESUME_RUN_DIR = None  # Example: OUTPUT_ROOT / "20260913T120000000000Z"

NUMBER_OF_QUESTIONS = None  # Positive integer = first N rows; None = all rows.
LIVE = True
MAX_WORKERS = 4
GENERATION_CONFIG = GenerationConfig(max_output_tokens=1024, timeout_seconds=60, provider="DeepInfra")
CONTEXT_BUDGET = ContextBudget(total_chars=20000, instruction_reserve=2000, answer_reserve=4000)

assert isinstance(MAX_WORKERS, int) and not isinstance(MAX_WORKERS, bool) and 1 <= MAX_WORKERS <= 16
print({
    "question_path": str(QUESTION_PATH.resolve()),
    "index": str(INDEX_DIR.resolve()),
    "output_root": str(OUTPUT_ROOT.resolve()),
    "number_of_questions": NUMBER_OF_QUESTIONS,
    "live": LIVE,
    "workers": MAX_WORKERS,
    "api_key_available": bool(openrouter_api_key()),
})

{'question_path': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\data\\questions\\Q_S1.json', 'index': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\03_retrieval_enhanced\\20260913T120112863801Z', 'output_root': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation', 'number_of_questions': None, 'live': True, 'workers': 4, 'api_key_available': True}


## Load and freeze inputs

The benchmark file is hashed before processing. Composite record keys use the filename and row ID, for example `Q_S1:1`. A live run stops before dispatch when the API key is unavailable.

In [3]:
benchmark = benchmark_identity(QUESTION_PATH)
selected_questions = select_questions(benchmark["questions"], NUMBER_OF_QUESTIONS)
if LIVE and not openrouter_api_key():
    raise RuntimeError("LIVE=True requires OPENROUTER_API_KEY; no requests were started.")

run_config = {
    "run_schema": "bulk-answer-run/v1",
    "dataset": benchmark["dataset"],
    "question_path": benchmark["path"],
    "benchmark_sha256": benchmark["sha256"],
    "available_question_count": benchmark["question_count"],
    "selected_question_count": len(selected_questions),
    "number_of_questions": NUMBER_OF_QUESTIONS,
    "index_dir": str(INDEX_DIR.resolve()),
    "retrieval_database_sha256": digest(INDEX_DIR / "retrieval.sqlite"),
    "passage_database_sha256": digest(INDEX_DIR / "passage.sqlite"),
    "enhancement_manifest_sha256": digest(INDEX_DIR / "enhancement_manifest.json"),
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": digest(PROMPT_PATH),
    "generation_config": asdict(GENERATION_CONFIG),
    "context_budget": asdict(CONTEXT_BUDGET),
    "live": LIVE,
    "max_workers": MAX_WORKERS,
}
print({
    "dataset": benchmark["dataset"],
    "available": benchmark["question_count"],
    "selected": len(selected_questions),
    "benchmark_sha256": benchmark["sha256"],
})

{'dataset': 'Q_S1', 'available': 100, 'selected': 100, 'benchmark_sha256': '8ca342ad54ecee63d357b6fb0034672dbd01fce73c0fbf84d2b8998e37207b93'}


## Create or resume the run

A new run writes its manifest before submitting work. Resume compares all result-affecting settings except `MAX_WORKERS`, which may be changed safely because it affects scheduling rather than individual request content.

In [4]:
if RESUME_RUN_DIR is None:
    RUN_DIR = OUTPUT_ROOT / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    write_json(RUN_DIR / "run_manifest.json", {**run_config, "created_at_utc": datetime.now(UTC).isoformat()})
else:
    RUN_DIR = Path(RESUME_RUN_DIR).resolve()
    saved_config = json.loads((RUN_DIR / "run_manifest.json").read_text(encoding="utf-8"))
    comparison_keys = [
        "dataset", "benchmark_sha256", "selected_question_count", "number_of_questions", "index_dir",
        "retrieval_database_sha256", "passage_database_sha256", "enhancement_manifest_sha256",
        "prompt_version", "prompt_sha256", "generation_config", "context_budget", "live",
    ]
    mismatches = [key for key in comparison_keys if saved_config.get(key) != run_config.get(key)]
    if mismatches:
        raise RuntimeError(f"Resume configuration differs for: {mismatches}")

RECORDS_PATH = RUN_DIR / "records.jsonl"
print("Actual run directory:", RUN_DIR)

Actual run directory: C:\Users\PK\Desktop\projects\mobile_rag\artifacts\05_2_bulk_answer_generation\20260913T124928720084Z


## Load checkpoints

JSONL is written with ASCII escaping so embedded Unicode line separators cannot split records. Every existing record must match the benchmark identity. Worker errors are retained as outcomes; they are not silently retried.

In [5]:
records_by_key = {}
if RECORDS_PATH.exists():
    with RECORDS_PATH.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            if record.get("dataset") != benchmark["dataset"] or record.get("benchmark_sha256") != benchmark["sha256"]:
                raise RuntimeError(f"Checkpoint identity mismatch on line {line_number}")
            key = record["record_key"]
            if key in records_by_key:
                raise RuntimeError(f"Duplicate checkpoint record: {key}")
            records_by_key[key] = record

pending = [
    row for row in selected_questions
    if f"{benchmark['dataset']}:{row['id']}" not in records_by_key
]
print({"already_completed": len(records_by_key), "pending": len(pending)})

{'already_completed': 0, 'pending': 100}


## Process pending questions in parallel

Each worker opens independent read-only SQLite connections and executes retrieval, context preparation, and generation. The main thread alone writes checkpoint records, flushing each result to disk before moving on.

In [6]:
batch_started = time.perf_counter()
if pending:
    with (
        RECORDS_PATH.open("a", encoding="utf-8", newline="\n") as checkpoint,
        ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool,
    ):
        futures = {
            pool.submit(
                process_question,
                row,
                dataset=benchmark["dataset"],
                benchmark_sha256=benchmark["sha256"],
                index_dir=INDEX_DIR,
                live=LIVE,
                generation_config=GENERATION_CONFIG,
                context_budget=CONTEXT_BUDGET,
            ): row
            for row in pending
        }
        for future in as_completed(futures):
            record = future.result()
            checkpoint.write(json.dumps(record, ensure_ascii=True, sort_keys=True) + "\n")
            checkpoint.flush()
            os.fsync(checkpoint.fileno())
            records_by_key[record["record_key"]] = record
            print(f"[{len(records_by_key)}/{len(selected_questions)}] {record['record_key']}: {record['pipeline_status']}")

batch_seconds = time.perf_counter() - batch_started
print({"newly_processed": len(pending), "batch_seconds": batch_seconds})

[1/100] Q_S1:2: answered
[2/100] Q_S1:1: answered
[3/100] Q_S1:3: answered
[4/100] Q_S1:4: answered
[5/100] Q_S1:7: answered
[6/100] Q_S1:6: answered
[7/100] Q_S1:5: answered
[8/100] Q_S1:8: answered
[9/100] Q_S1:9: answered
[10/100] Q_S1:10: answered
[11/100] Q_S1:11: answered
[12/100] Q_S1:13: answered
[13/100] Q_S1:12: answered
[14/100] Q_S1:14: answered
[15/100] Q_S1:15: answered
[16/100] Q_S1:16: answered
[17/100] Q_S1:17: answered
[18/100] Q_S1:18: answered
[19/100] Q_S1:19: answered
[20/100] Q_S1:21: answered
[21/100] Q_S1:20: answered
[22/100] Q_S1:22: answered
[23/100] Q_S1:24: answered
[24/100] Q_S1:23: answered
[25/100] Q_S1:25: answered
[26/100] Q_S1:26: answered
[27/100] Q_S1:28: answered
[28/100] Q_S1:27: answered
[29/100] Q_S1:30: answered
[30/100] Q_S1:29: answered
[31/100] Q_S1:31: answered
[32/100] Q_S1:32: answered
[33/100] Q_S1:33: answered
[34/100] Q_S1:34: answered
[35/100] Q_S1:37: answered
[36/100] Q_S1:35: answered
[37/100] Q_S1:36: answered
[38/100] Q_S1:38: a

## Save the compact review table and summary

Full evidence remains in `records.jsonl`. The CSV extracts practical fields for later evaluation. API errors, invalid responses, abstentions, and missing records remain visible rather than being removed from totals.

In [7]:
ordered_keys = [f"{benchmark['dataset']}:{row['id']}" for row in selected_questions]
ordered_records = [records_by_key[key] for key in ordered_keys if key in records_by_key]
overview = []
for record in ordered_records:
    row = record["question_record"]
    generated = record.get("generation") or {}
    answer = generated.get("answer") or {}
    context = record.get("context") or {}
    retrieval = record.get("retrieval") or {}
    usage = generated.get("usage") or {}
    overview.append({
        "record_key": record["record_key"],
        "question_id": row.get("id"),
        "category": row.get("category"),
        "topic": row.get("topic"),
        "question": row.get("question"),
        "reference_answer": row.get("answer"),
        "source_of_truth": row.get("source_of_truth"),
        "pipeline_status": record["pipeline_status"],
        "answer_status": answer.get("status"),
        "generated_answer": answer.get("answer"),
        "generated_reason": answer.get("reason"),
        "citations": json.dumps(answer.get("citations"), ensure_ascii=False),
        "retrieval_status": retrieval.get("status"),
        "retrieval_hits": len(retrieval.get("hits", [])),
        "context_status": context.get("status"),
        "context_groups": len(context.get("evidence_groups", [])),
        "context_characters": (context.get("budget") or {}).get("used"),
        "provider": generated.get("provider"),
        "returned_model": generated.get("returned_model"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "generation_latency_ms": generated.get("latency_ms"),
        "pipeline_seconds": record.get("pipeline_seconds"),
    })

if overview:
    with (RUN_DIR / "results.csv").open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(overview[0]))
        writer.writeheader()
        writer.writerows(overview)

status_counts = Counter(record["pipeline_status"] for record in ordered_records)
timings = [
    record["pipeline_seconds"] for record in ordered_records
    if isinstance(record.get("pipeline_seconds"), (int, float))
]
missing_keys = [key for key in ordered_keys if key not in records_by_key]
summary = {
    "run_schema": "bulk-answer-summary/v1",
    "run_dir": str(RUN_DIR.resolve()),
    "dataset": benchmark["dataset"],
    "benchmark_sha256": benchmark["sha256"],
    "available_questions": benchmark["question_count"],
    "selected_questions": len(selected_questions),
    "saved_records": len(ordered_records),
    "complete": not missing_keys,
    "missing_record_keys": missing_keys,
    "pipeline_status_counts": dict(sorted(status_counts.items())),
    "live": LIVE,
    "max_workers": MAX_WORKERS,
    "last_session_seconds": batch_seconds,
    "pipeline_seconds": {
        "median": statistics.median(timings) if timings else None,
        "p95": sorted(timings)[round((len(timings) - 1) * 0.95)] if timings else None,
        "samples": len(timings),
    },
    "updated_at_utc": datetime.now(UTC).isoformat(),
}
write_json(RUN_DIR / "summary.json", summary)
summary

{'run_schema': 'bulk-answer-summary/v1',
 'run_dir': 'C:\\Users\\PK\\Desktop\\projects\\mobile_rag\\artifacts\\05_2_bulk_answer_generation\\20260913T124928720084Z',
 'dataset': 'Q_S1',
 'benchmark_sha256': '8ca342ad54ecee63d357b6fb0034672dbd01fce73c0fbf84d2b8998e37207b93',
 'available_questions': 100,
 'selected_questions': 100,
 'saved_records': 100,
 'complete': True,
 'missing_record_keys': [],
 'pipeline_status_counts': {'answered': 100},
 'live': True,
 'max_workers': 4,
 'last_session_seconds': 130.4392128000036,
 'pipeline_seconds': {'median': 5.1991060000727884,
  'p95': 6.841273800004274,
  'samples': 100},
 'updated_at_utc': '2026-09-13T12:51:39.249455+00:00'}

## Inspect one complete record

The record keeps the question, reference answer, retrieved evidence, prepared context, generated answer, citations, usage, and timings together. Structural validity does not establish clinical correctness.

In [8]:
if ordered_records:
    sample = ordered_records[0]
    display({
        "record_key": sample["record_key"],
        "question": sample["question_record"]["question"],
        "reference_answer": sample["question_record"].get("answer"),
        "generation": sample.get("generation"),
        "context": sample.get("context"),
    })

{'record_key': 'Q_S1:1',
 'question': 'A frontline health worker asks: Emergency triage signs. What is the correct action or answer?',
 'reference_answer': 'Obstructed or absent breathing, severe respiratory distress, central cyanosis, shock, coma/reduced consciousness, convulsions, or severe dehydration with diarrhoea require immediate emergency treatment.',
 'generation': {'status': 'answered',
  'model': 'google/gemma-3-4b-it',
  'prompt_version': 'evidence-answer/v2',
  'config': {'max_output_tokens': 1024,
   'timeout_seconds': 60,
   'provider': 'DeepInfra'},
  'answer': {'status': 'answered',
   'answer': 'Emergency triage signs include obstructed or absent breathing, severe respiratory distress, central cyanosis, signs of shock, coma, convulsions, and signs of severe dehydration in a child with diarrhea. If any of these signs are present, the frontline health worker should call for help, assess and resuscitate, give treatment, and draw blood for emergency laboratory investigati